# Train on Kaggle (free GPU)

1. Create a new Kaggle notebook → **Add data** → competition *rsna-pneumonia-detection-challenge*.
2. Settings → Accelerator **GPU T4 x2** (or P100), Internet **On**.
3. Upload this repo as a Kaggle *Dataset* (zip it, without `data/` and `models/`) named `medimg-capstone`, and add it too — or `git clone` it if it is on GitHub.
4. Run all cells (~1–2 h for 12 epochs at 320 px). Download `models/classifier.pt`, `metrics/`, `reports/` from the Output tab.

In [ ]:
import os, shutil, subprocess
SRC = '/kaggle/input/medimg-capstone/medimg-capstone'   # or: !git clone https://github.com/<you>/medimg-capstone
WORK = '/kaggle/working/medimg-capstone'
if not os.path.exists(WORK):
    shutil.copytree(SRC, WORK)
os.chdir(WORK)
!pip install -q -e ./ml[train]

In [ ]:
# Point the prepare stage at the competition data instead of downloading 4 GB
os.makedirs('data/raw', exist_ok=True)
if not os.path.exists('data/raw/rsna'):
    os.symlink('/kaggle/input/rsna-pneumonia-detection-challenge', 'data/raw/rsna')
!ls data/raw/rsna | head

In [ ]:
!python ml/scripts/prepare_rsna.py

In [ ]:
import torch; print(torch.cuda.get_device_name(0))
!python ml/scripts/train_classifier.py

In [ ]:
!python ml/scripts/evaluate_classifier.py
from IPython.display import Image, Markdown, display
display(Image('reports/roc.png')); display(Markdown(open('reports/eval_cls.md').read()))

## Look at the heatmaps yourself
Review false positives and false negatives — this error analysis is the most valuable chapter of your report.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image as PImage
from medai.models import load_checkpoint
from medai.predict import classify
model, meta = load_checkpoint('models/classifier.pt')
df = pd.read_csv('data/processed/rsna/metadata.csv'); te = df[df.split=='test'].sample(8, random_state=1)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, r in zip(ax.flat, te.itertuples()):
    img = np.asarray(PImage.open(f'data/processed/rsna/images/{r.patientId}.png').convert('L'), np.float32)/255
    res = classify(model, meta, img)
    a.imshow(img, cmap='gray'); a.imshow(res.cam, cmap='jet', alpha=0.35)
    a.set_title(f'GT={r.label}  p={res.top_probability:.2f}  {r.view}'); a.axis('off')
plt.tight_layout()

In [ ]:
# Optional: 3D U-Net spleen segmentation (~1-2 h). Downloads MSD Task09 (1.5 GB).
# !python ml/scripts/train_segmentation.py